# 🎓 PRÁTICA: DATASETS E DATALOADERS COM PyTorch

## Entendendo como processar dados ANTES de treinar modelos

### Objetivo
- ✅ O que são Tensores PyTorch
- ✅ Como criar Datasets customizados
- ✅ Como usar DataLoaders
- ✅ Transformações com torchvision
- ✅ **SEM treinar nenhuma rede neural**

## 📦 Setup e Verificação

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'✓ PyTorch versão: {torch.__version__}')
print(f'✓ Torchvision versão: {torchvision.__version__}')
print(f'✓ NumPy versão: {np.__version__}')
print(f'\n📱 GPU/Dispositivos:')
print(f'  GPU disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

---
# PARTE 1: DATASET SIMPLES COM TENSORES PyTorch

## Passo 1: Criar dados simples

In [ ]:
n_samples = 100
n_features = 5

X = np.random.randn(n_samples, n_features).astype(np.float32)
y = np.random.randint(0, 2, n_samples).astype(np.int64)

print('📊 Dados criados:')
print(f'  Features shape: {X.shape}')
print(f'  Labels shape: {y.shape}')
print(f'  Features range: {X.min():.2f} a {X.max():.2f}')
print(f'  Classes: {np.unique(y)}')

## Passo 2: Converter para Tensores PyTorch

In [ ]:
X_tensor = torch.from_numpy(X)
y_tensor = torch.from_numpy(y)

print('🔧 Tensores PyTorch:')
print(f'  Tipo X_tensor: {type(X_tensor)}')
print(f'  Dtype: {X_tensor.dtype}')
print(f'  Shape: {X_tensor.shape}')
print(f'  Device: {X_tensor.device}')

print(f'\n📈 Operações com tensores:')
print(f'  Média do primeiro feature: {X_tensor[:, 0].mean():.4f}')
print(f'  Máximo valor: {X_tensor.max():.4f}')
print(f'  Mínimo valor: {X_tensor.min():.4f}')

## Passo 3: Criar Dataset Customizado

In [ ]:
from torch.utils.data import Dataset, DataLoader

class DatasetSimples(Dataset):
    def __init__(self, X, y):
        ...

    def __len__(self):
        ...

    def __getitem__(self, idx):
        ...

dataset = ...

print(f'📦 Dataset criado:')
print(f'  Total de amostras: {len(dataset)}')

features, label = dataset[0]
print(f'\n🔍 Uma amostra:')
print(f'  Features: {features}')
print(f'  Label: {label}')

## Passo 4: Criar DataLoader

In [ ]:
batch_size = 10
dataloader = ...

print(f'📦 DataLoader criado:')
print(f'  Batch size: {batch_size}')
print(f'  Total de batches: {len(dataloader)}')

for X_batch, y_batch in dataloader:
    print(f'\n🔍 Um batch:')
    print(f'  Shape features: {X_batch.shape}')
    print(f'  Shape labels: {y_batch.shape}')
    break

---
# PARTE 2: DATASET REAL COM IMAGENS (CIFAR-10)

## Passo 1: Carregar CIFAR-10

In [ ]:
from torchvision import datasets, transforms

transform_simples = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print('📥 Carregando CIFAR-10...')
cifar10_train = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_simples
)

print(f'✓ Dataset carregado!')
print(f'  Treino: {len(cifar10_train)} imagens')
print(f'  Classes: {cifar10_train.classes}')

## Passo 2: Visualizar amostras de cada classe

In [ ]:
def desnormalizar(tensor):
    means = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
    stds = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
    tensor = tensor * stds + means
    return torch.clamp(tensor, 0, 1)

class_names = cifar10_train.classes

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('CIFAR-10: Uma amostra de cada classe', fontsize=14, fontweight='bold')

for class_idx in range(10):
    idx = 0
    for i, (img, label) in enumerate(cifar10_train):
        if label == class_idx:
            idx = i
            break

    img, label = cifar10_train[idx]
    img_vis = desnormalizar(img).permute(1, 2, 0).numpy()

    ax = axes[class_idx // 5, class_idx % 5]
    ax.imshow(img_vis)
    ax.set_title(f'{class_names[label]}')
    ax.axis('off')

plt.tight_layout()
plt.show()
print('✓ Visualização concluída!')

## Passo 3: Criar DataLoader para CIFAR-10

In [ ]:
dataloader_cifar = ...

print(f'✓ DataLoader criado')
print(f'  Batch size: 32')
print(f'  Total de batches: {len(dataloader_cifar)}')

for X_batch, y_batch in dataloader_cifar:
    print(f'\n🔍 Um batch:')
    print(f'  Shape: {X_batch.shape}')
    print(f'  Labels: {y_batch[:8].tolist()}')
    break

## Passo 4: Transformações

In [ ]:
transform_aumentado = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print('✓ Transformações definidas:')
print('  • Flip horizontal aleatório')
print('  • ColorJitter (brilho, contraste, saturação)')
print('  • Normalização')

## Passo 5: Visualizar Batch

In [ ]:
for X_batch, y_batch in dataloader_cifar:
    fig = plt.figure(figsize=(14, 8))
    fig.suptitle('CIFAR-10: Um Batch do DataLoader', fontsize=14, fontweight='bold')

    for i in range(16):
        ax = plt.subplot(4, 4, i + 1)
        imagem = desnormalizar(X_batch[i]).permute(1, 2, 0).numpy()
        label = y_batch[i].item()

        ax.imshow(imagem)
        ax.set_title(f'{class_names[label]}', fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    break

---
# 📊 RESUMO

In [ ]:
print('='*70)
print('✅ PRÁTICA CONCLUÍDA COM SUCESSO!')
print('='*70)
print()
print('O que você aprendeu:')
print('  ✓ Criar tensores PyTorch')
print('  ✓ Dataset customizado')
print('  ✓ DataLoader com paralelização')
print('  ✓ Transformações com torchvision')
print('  ✓ Processamento eficiente de dados')
print()
print('Próximos passos:')
print('  → Treinar uma CNN')
print('  → Usar modelos pré-treinados')
print('  → Transfer Learning')
print('='*70)